# Social Behavior Processing Pipeline

**Project:** Social Single/Group Housing Behavior Analysis  
**Author:** Mir Qi  
**Last Updated:** October 20, 2024

## Overview
This notebook processes and validates multi-camera behavioral recordings for social interaction experiments. The pipeline includes:
1. Scanning and logging folder structures
2. Reading and consolidating parquet logs
3. Data quality checks and filtering
4. Visualization of COM (center of mass) trajectories
5. DANNCE 3D pose validation

---
## 1. Scan Folders & Generate Logs

In [4]:
import sys
import os
sys.path.append(os.path.abspath('../'))

from status_fields.status_fields_config_oct3v1_brws_250525 import STATUS_FIELDS_CONFIG
from utlis.scan_engine_utlis.scan_eng_big_utlis import log_folder_to_parquet_sep

base_folder = "/data/big_rim/rsync_dcc_sum/25Nov"
failed_paths_file = None
force_rescan_rec_files = []
rescan_threshold_days = 0.000000001

log_folder_to_parquet_sep(base_folder, failed_paths_file, STATUS_FIELDS_CONFIG,
                            force_rescan_rec_files=force_rescan_rec_files,
                            rescan_threshold_days=rescan_threshold_days)

Log for ZIcI1_1mW saved at /data/big_rim/rsync_dcc_sum/25Nov/2025_10_31/ZIcI1_1mW/folder_log.parquet
Log for ZIcI1_00mW saved at /data/big_rim/rsync_dcc_sum/25Nov/2025_10_31/ZIcI1_00mW/folder_log.parquet
Log for ZIcI1_0mW saved at /data/big_rim/rsync_dcc_sum/25Nov/2025_10_31/ZIcI1_0mW/folder_log.parquet
Log for ZIcI1_10mW saved at /data/big_rim/rsync_dcc_sum/25Nov/2025_10_31/ZIcI1_10mW/folder_log.parquet
Log for ZIcI1_5mW saved at /data/big_rim/rsync_dcc_sum/25Nov/2025_10_31/ZIcI1_5mW/folder_log.parquet


---
## 2. Load All Data

In [5]:
sys.path.append(os.path.abspath('../..'))
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files

all_df = read_all_parquet_files(base_folder)

---
## 3. Check Schema & View Data

In [6]:
print("Schema:")
print(all_df.schema)
print("\n" + "="*50 + "\n")
print(all_df.to_pandas())

Schema:
mir_generate_param: string
sync: string
mini_6cam_map: string
dropf_handle: string
com: string
com_vis: string
social: string
miniscope: string
test: string
after_oxytocin: string
before_oxytocin: string
dannce: string
dannce_vis: string
mini_rec_sync: string
mini_rec_sync_com: string
rec_file: string
scan_time: string
rec_path: string
date_folder: string
calib_files: list<element: string>
  child 0, element: string
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 2661


  mir_generate_param sync mini_6cam_map dropf_handle com com_vis social  \
0                  1    1             0            0   1       1      0   
1                  1    1             0            0   1       1      0   
2                  1    1             0            0   1       1      0   
3                  1    1             0            0   1       1      0   
4                  1    1             0            0   1       1      0   
5               

In [7]:
all_df.to_pandas().columns.tolist()

['mir_generate_param',
 'sync',
 'mini_6cam_map',
 'dropf_handle',
 'com',
 'com_vis',
 'social',
 'miniscope',
 'test',
 'after_oxytocin',
 'before_oxytocin',
 'dannce',
 'dannce_vis',
 'mini_rec_sync',
 'mini_rec_sync_com',
 'rec_file',
 'scan_time',
 'rec_path',
 'date_folder',
 'calib_files']

---
## 4. Filter Sessions

**Edit the conditions list below to filter your data.**

In [8]:
import pyarrow.compute as pc
from functools import reduce

table = all_df

conditions = [
    pc.equal(table['social'], '1'),
    pc.equal(table['com'], '1'),
    pc.equal(table['com_vis'], '0'),
]

filter_mask = reduce(pc.and_, conditions)
filtered_table = table.filter(filter_mask)

print(f"Filtered: {len(filtered_table)} sessions")
print(filtered_table.to_pandas())

Filtered: 0 sessions
Empty DataFrame
Columns: [mir_generate_param, sync, mini_6cam_map, dropf_handle, com, com_vis, social, miniscope, test, after_oxytocin, before_oxytocin, dannce, dannce_vis, mini_rec_sync, mini_rec_sync_com, rec_file, scan_time, rec_path, date_folder, calib_files]
Index: []


### Common Filter Combinations

Uncomment the one you need:

In [9]:
# For COM processing: social=1, com=0
# conditions = [
#     pc.equal(table['social'], '1'),
#     pc.equal(table['com'], '0'),
# ]

# For COM visualization: social=1, com=1, com_vis=0
# conditions = [
#     pc.equal(table['social'], '1'),
#     pc.equal(table['com'], '1'),
#     pc.equal(table['com_vis'], '0'),
# ]

# For DANNCE validation: dannce=1, dannce_vis=0
# conditions = [
#     pc.equal(table['dannce'], '1'),
#     pc.equal(table['dannce_vis'], '0'),
# ]

# For sync processing: sync=0
# conditions = [
#     pc.equal(table['sync'], '0'),
# ]

# For specific date:
# conditions = [
#     pc.equal(table['date_folder'], '2025_10_03'),
# ]

filter_mask = reduce(pc.and_, conditions)
filtered_table = table.filter(filter_mask)
print(f"Filtered: {len(filtered_table)} sessions")

Filtered: 0 sessions


---
## 5. Visualize Timeline

In [10]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from datetime import datetime

filtered_df = filtered_table.to_pandas()

full_paths = [
    os.path.join(base_folder, date_folder, rec_file)
    for date_folder, rec_file in zip(filtered_df['date_folder'], filtered_df['rec_file'])
]

sessions = []
for p in full_paths:
    ft = os.path.join(p, "videos", "Camera1", "frametimes.mat")
    if os.path.isfile(ft):
        mtime = os.path.getmtime(ft)
        sessions.append((p, mtime))

sessions.sort(key=lambda x: x[1])

if len(sessions) > 0:
    n = len(sessions)
    fig, axes = plt.subplots(1, n, figsize=(n * 3, 3))
    if n == 1:
        axes = [axes]

    for ax, (p, mtime) in zip(axes, sessions):
        img_file = os.path.join(p, "COM/predict00/vis", "com_circle.png")
        if os.path.isfile(img_file):
            img = mpimg.imread(img_file)
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "no image", ha="center", va="center")
        
        ts = datetime.fromtimestamp(mtime).strftime("%Y-%m-%d %H:%M:%S")
        ax.set_title(ts, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No sessions to visualize")

No sessions to visualize


---
## 6. Process COM Visualizations

In [11]:
from utlis.vis_valid_utlis.scom_traga_utlis import plot_com_all_social

for_com_vis = filtered_table

records = [
    {
        'date_folder': date_folder.as_py(),
        'rec_file': rec_file.as_py()
    }
    for date_folder, rec_file in zip(for_com_vis['date_folder'], for_com_vis['rec_file'])
]

print(f"Processing {len(records)} sessions...\n")

for i, record in enumerate(records, 1):
    base_path = f"{base_folder}/{record['date_folder']}/{record['rec_file']}"
    print(f"[{i}/{len(records)}] {base_path}")
    
    try:
        plot_com_all_social(base_path, perform_generate_com_video=True)
        print("  ✓ Complete")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        continue

print("\n=== COM visualization complete ===")

Processing 0 sessions...


=== COM visualization complete ===


---
## 7. DANNCE Validation

In [12]:
from useful_files.sophie_check_dannce_mir_modif import dannce_valid
from concurrent.futures import ProcessPoolExecutor, as_completed

for_dannce_vis = filtered_table

records = [
    {
        'date_folder': date_folder.as_py(),
        'rec_file': rec_file.as_py()
    }
    for date_folder, rec_file in zip(for_dannce_vis['date_folder'], for_dannce_vis['rec_file'])
]

print(f"Processing {len(records)} DANNCE sessions...\n")

def process_record(record):
    base_path = f"{base_folder}/{record['date_folder']}/{record['rec_file']}"
    print(base_path)
    try:
        dannce_valid(base_path)
        return f"✓ {base_path}"
    except Exception as e:
        return f"✗ {base_path}: {e}"

with ProcessPoolExecutor() as executor:
    futures = [executor.submit(process_record, record) for record in records]
    for future in as_completed(futures):
        result = future.result()
        print(result)

print("\n=== DANNCE validation complete ===")

Processing 0 DANNCE sessions...


=== DANNCE validation complete ===


---
## Notes

### Filter Syntax
```python
conditions = [
    pc.equal(table['column'], 'value'),  # Use strings for status fields
]
filter_mask = reduce(pc.and_, conditions)
filtered_table = table.filter(filter_mask)
```

### Troubleshooting
If you get `ArrowNotImplementedError` about types not matching:
1. Run the schema check cell to see column types
2. Make sure you're comparing with the correct type (usually strings like '0' or '1')
3. If a column is a list type, you may need different filtering logic